In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy scikit-learn tabulate

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
import time
import os

print("--- CHUẨN BỊ DỮ LIỆU ĐÁNH GIÁ ---")

# 1. THIẾT LẬP ĐƯỜNG DẪN (Tùy chỉnh BASE_PATH theo đúng thư mục Drive của bạn)
BASE_PATH = "/content/drive/MyDrive/ĐATN/data"
META_CSV = f"{BASE_PATH}/processed/metadata/df_multimodal_prompts.csv"
VECTOR_DIR = f"{BASE_PATH}/processed/embeddings/text_features"

# 2. TẢI METADATA VÀ PHÂN TÁCH QUERY/GALLERY
df = pd.read_csv(META_CSV)
product_ids = df['product_id'].values
partitions = df['partition'].values

# Lấy index (vị trí dòng) của tập Query và Gallery
query_indices = np.where(partitions == 'query')[0]
gallery_indices = np.where(partitions == 'gallery')[0]

query_labels = product_ids[query_indices]
gallery_labels = product_ids[gallery_indices]

print(f"Tổng số truy vấn (Query): {len(query_indices)}")
print(f"Tổng số sản phẩm trong kho (Gallery): {len(gallery_indices)}\n")

--- CHUẨN BỊ DỮ LIỆU ĐÁNH GIÁ ---
Tổng số truy vấn (Query): 1000
Tổng số sản phẩm trong kho (Gallery): 11278



hàm tính toán Metrics

In [3]:
def evaluate_retrieval(query_vectors, gallery_vectors, q_labels, g_labels, k_list=[1, 5, 10]):
    # Chuẩn hóa độ dài vector về 1 (L2 Norm) để dùng phép nhân ma trận thay cho Cosine
    query_vectors = normalize(query_vectors, axis=1)
    gallery_vectors = normalize(gallery_vectors, axis=1)

    # Tính ma trận tương đồng (Shape: 1000 Query x 11278 Gallery)
    sim_matrix = np.dot(query_vectors, gallery_vectors.T)

    # Lấy index của Top K phần tử có độ tương đồng lớn nhất
    max_k = max(k_list)
    top_k_indices = np.argsort(-sim_matrix, axis=1)[:, :max_k]

    metrics = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_list}

    for i in range(len(q_labels)):
        q_label = q_labels[i]
        retrieved_idx = top_k_indices[i]
        retrieved_labels = g_labels[retrieved_idx]

        # Đếm tổng số sản phẩm đúng có trong Gallery
        relevant_total = np.sum(g_labels == q_label)
        if relevant_total == 0:
            continue

        for k in k_list:
            retrieved_k = retrieved_labels[:k]
            hits_k = (retrieved_k == q_label).astype(int) # Mảng 0, 1 (1 là trúng đích)
            hits_count = np.sum(hits_k)

            # Tính Metrics
            precision = hits_count / k
            recall = hits_count / relevant_total

            dcg = np.sum(hits_k / np.log2(np.arange(2, k + 2)))
            ideal_hits = np.zeros(k)
            ideal_hits[:min(k, relevant_total)] = 1
            idcg = np.sum(ideal_hits / np.log2(np.arange(2, k + 2)))
            ndcg = dcg / idcg if idcg > 0 else 0

            metrics[k]['precision'].append(precision)
            metrics[k]['recall'].append(recall)
            metrics[k]['ndcg'].append(ndcg)

    # Tính điểm trung bình của toàn bộ tập Query
    results = {}
    for k in k_list:
        results[f'P@{k}'] = np.mean(metrics[k]['precision'])
        results[f'R@{k}'] = np.mean(metrics[k]['recall'])
        results[f'NDCG@{k}'] = np.mean(metrics[k]['ndcg'])

    return results
print("Đã khởi tạo hàm evaluate_retrieval thành công!")

Đã khởi tạo hàm evaluate_retrieval thành công!


đánh giá cho toàn bộ 4 mô hình

In [4]:
# DANH SÁCH CÁC MÔ HÌNH CẦN ĐÁNH GIÁ
models = {
    "TF-IDF Baseline": "tfidf_embeddings.npy",
    "FastText Baseline": "fasttext_embeddings.npy",
    "SBERT (NLP SOTA)": "sbert_embeddings.npy",
    "Fashion-CLIP (Text-only)": "fashionclip_text_embeddings.npy"
}

K_VALUES = [1, 5, 10]
evaluation_results = []

print("--- BẮT ĐẦU CHẠY ĐÁNH GIÁ (EVALUATION) ---")
for model_name, file_name in models.items():
    print(f"Đang xử lý: {model_name}...")
    start_time = time.time()
    try:
        file_path = f"{VECTOR_DIR}/{file_name}"
        if not os.path.exists(file_path):
            print(f"Lỗi: Không tìm thấy file {file_name}")
            continue

        all_vectors = np.load(file_path)

        # Tách vector thành tập Query và Gallery
        q_vecs = all_vectors[query_indices]
        g_vecs = all_vectors[gallery_indices]

        # Tính toán
        res = evaluate_retrieval(q_vecs, g_vecs, query_labels, gallery_labels, k_list=K_VALUES)
        res['Model'] = model_name
        evaluation_results.append(res)

        elapsed_time = time.time() - start_time
        print(f"Xong {model_name} (mất {elapsed_time:.2f}s). R@10: {res['R@10']:.4f}, NDCG@10: {res['NDCG@10']:.4f}")

    except Exception as e:
        print(f"Lỗi hệ thống khi chạy {model_name}: {e}")

--- BẮT ĐẦU CHẠY ĐÁNH GIÁ (EVALUATION) ---
Đang xử lý: TF-IDF Baseline...
Xong TF-IDF Baseline (mất 2.64s). R@10: 0.1132, NDCG@10: 0.0795
Đang xử lý: FastText Baseline...
Xong FastText Baseline (mất 2.76s). R@10: 0.0431, NDCG@10: 0.0286
Đang xử lý: SBERT (NLP SOTA)...
Xong SBERT (NLP SOTA) (mất 2.85s). R@10: 0.0873, NDCG@10: 0.0594
Đang xử lý: Fashion-CLIP (Text-only)...
Xong Fashion-CLIP (Text-only) (mất 3.90s). R@10: 0.3504, NDCG@10: 0.2425


ket qua

In [5]:
print("\n" + "="*85)
print("BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ (TEXT-TO-TEXT RETRIEVAL)")
print("="*85)

df_results = pd.DataFrame(evaluation_results)

# Sắp xếp lại thứ tự cột
cols = ['Model']
for k in K_VALUES:
    cols.extend([f'P@{k}', f'R@{k}', f'NDCG@{k}'])
df_results = df_results[cols]

# In dạng bảng Markdown (Làm tròn 4 chữ số thập phân)
print(df_results.round(4).to_markdown(index=False))

# Lưu file kết quả CSV ra Drive
output_csv = f"{BASE_PATH}/processed/evaluation_text_only_baselines.csv"
df_results.to_csv(output_csv, index=False)
print(f"\n💾 Đã lưu bảng kết quả thô tại: {output_csv}")


BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ (TEXT-TO-TEXT RETRIEVAL)
| Model                    |   P@1 |    R@1 |   NDCG@1 |    P@5 |    R@5 |   NDCG@5 |   P@10 |   R@10 |   NDCG@10 |
|:-------------------------|------:|-------:|---------:|-------:|-------:|---------:|-------:|-------:|----------:|
| TF-IDF Baseline          | 0.06  | 0.0361 |    0.06  | 0.0278 | 0.0841 |   0.0694 | 0.019  | 0.1132 |    0.0795 |
| FastText Baseline        | 0.019 | 0.0117 |    0.019 | 0.0098 | 0.0317 |   0.0245 | 0.0071 | 0.0431 |    0.0286 |
| SBERT (NLP SOTA)         | 0.04  | 0.0253 |    0.04  | 0.02   | 0.0596 |   0.0491 | 0.0153 | 0.0873 |    0.0594 |
| Fashion-CLIP (Text-only) | 0.158 | 0.1084 |    0.158 | 0.0826 | 0.2627 |   0.2108 | 0.0568 | 0.3504 |    0.2425 |

💾 Đã lưu bảng kết quả thô tại: /content/drive/MyDrive/ĐATN/data/processed/evaluation_text_only_baselines.csv
